# Activation Caching & Linear Probing

Cache activations to avoid redundant forward passes, and train linear probes to test
what information is linearly accessible at different layers.

In [ ]:
import interpkit

model = interpkit.load("gpt2")

## Activation Caching

When exploring the same input with multiple operations, caching avoids re-running the forward pass.

In [ ]:
# Cache activations at all layers for a given input
model.cache("The capital of France is")

print(f"Cache populated: {model.cached}")
print(f"Cached modules: {len(model._cache)}")

In [ ]:
# Now these are instant — no forward pass needed
act_0 = model.activations("The capital of France is", at="transformer.h.0.mlp")
act_8 = model.activations("The capital of France is", at="transformer.h.8.mlp")
act_11 = model.activations("The capital of France is", at="transformer.h.11.mlp")

print(f"Layer 0 norm:  {act_0.float().norm():.2f}")
print(f"Layer 8 norm:  {act_8.float().norm():.2f}")
print(f"Layer 11 norm: {act_11.float().norm():.2f}")

In [ ]:
# Cache auto-invalidates when the input changes
model.cache("A completely different sentence")
print(f"Cache still valid for original input: {model._cache_input_hash}")

# Or clear manually
model.clear_cache()
print(f"Cache cleared: {model.cached}")

## Selective caching

Cache only specific modules to save memory.

In [ ]:
model.cache(
    "The capital of France is",
    at=["transformer.h.0.mlp", "transformer.h.8.mlp", "transformer.h.11.mlp"],
)
print(f"Cached {len(model._cache)} modules")
model.clear_cache()

## Linear Probing

Train a linear classifier on activations to test what information is linearly separable at a given layer.

Here we probe whether GPT-2 can distinguish animal-related from object-related sentences.

In [ ]:
texts = [
    # Animals (label 0)
    "The cat sat on the mat",
    "The dog ran through the park",
    "A bird flew over the house",
    "The fish swam in the pond",
    "A horse galloped across the field",
    "The rabbit hopped through the garden",
    # Objects (label 1)
    "The car drove down the road",
    "A book sat on the shelf",
    "The lamp illuminated the room",
    "A chair stood by the window",
    "The phone rang on the desk",
    "A cup of coffee on the table",
]
labels = [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]

In [ ]:
# Probe at an early layer
result = model.probe(texts, labels, at="transformer.h.2")
print(f"\nLayer 2 accuracy: {result['accuracy']:.3f}")

In [ ]:
# Probe at a later layer
result = model.probe(texts, labels, at="transformer.h.10")
print(f"\nLayer 10 accuracy: {result['accuracy']:.3f}")

## Probe across all layers

See how linear separability evolves through the network.

In [ ]:
accuracies = []
for i in range(12):
    r = model.probe(texts, labels, at=f"transformer.h.{i}")
    accuracies.append(r["accuracy"])
    print(f"Layer {i:2d}: accuracy = {r['accuracy']:.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(range(12), accuracies, color="#53d769")
plt.xlabel("Layer")
plt.ylabel("Probe Accuracy")
plt.title("Linear Probe Accuracy Across Layers")
plt.ylim(0, 1.05)
plt.xticks(range(12))
plt.tight_layout()
plt.show()